# Moment 3 (part A): long_term_share, and (part B) the SA separation rate

Two outputs from one notebook, per the build plan: `long_term_share` is the fourth `moments.csv`
row, and the separation rate `lambda` is a model *parameter*, not a calibration target -- it's
directly estimated here rather than searched over in Week 4's MSM, the same status
`discount_rate` already has.

## Part A: long_term_share

QLFS's own `Long_term_unempl` classifies every currently-unemployed respondent as long-term
(>= 12 months searching) or short-term, matching `moments.csv`'s definition exactly: the share
of the *currently unemployed* with an in-progress spell of a year or more. Same four quarters,
same person-weighting, same Kish approximate-design-effect SE, and the same
most-recent-quarter-as-headline convention as notebook 01 -- no new methodology here, just the
next pre-built QLFS classification in the same pattern.

## Part B: the SA separation rate

`configs/baseline.yaml` currently carries `separation_rate: 0.0048`, Miyamoto (2011)'s Japanese
monthly rate, explicitly a fallback pending this notebook. The plan's own words: computed "if
the SA rate is not computable by end of week 3." It's computable.

QLFS is a **rotating panel** -- each sampled household is surveyed for several consecutive
quarters before rotating out, and PALMS's own household/person identifiers are only meaningful
*within* a single wave's rotation, not automatically across PALMS's 30-year harmonised span (see
`DECISIONS.md`'s notebook 01/02 entries for the same "don't trust an identifier further than it's
documented to reach" caution). But the raw QLFS quarterly files carry `UQNO` (household) and
`PERSONNO` (within-household), and consecutive quarters really do share sampled households:
linking 2025 Q2 to 2025 Q3 by `UQNO`+`PERSONNO` recovers 43,796 matched individuals out of
65,443 in Q2 -- a genuine ~67 per cent panel retention, not a coincidence at that scale.

**Method**: for each of the three available consecutive quarter-pairs (2025 Q2->Q3, Q3->Q4,
Q4->2026 Q1), take everyone matched across both quarters who was `Employed` in the first quarter,
and check whether they're still `Employed` in the second. The (weighted) share who aren't is that
pair's quarterly separation rate.

**Corrected 2026-08-17, following an independent verification.** An audit of this notebook found
implausible matches inside the raw `UQNO`+`PERSONNO` linkage: a household code being reassigned
to a different person between quarters produces a spurious "employment transition" that's really
a person-identity error, not a real separation. The linkage audit below is expanded to check
recorded gender and age for consistency across each matched pair, rejecting pairs that fail
either check; the design variables available for a proper survey-design uncertainty estimate are
audited explicitly rather than assumed; and the uncertainty estimate itself is redone as a
household-cluster bootstrap (resampling whole households, not individuals, so people in the same
household aren't treated as independent draws) rather than the individual-record bootstrap used
before. See DECISIONS.md, "The QLFS separation-rate linkage needed a demographic consistency
check, and the design variables needed auditing before trusting a plain bootstrap."

**Sanity-checked against StatsSA's own published panel transition figures**, not just trusted
because the computation ran without error: StatsSA reports quarterly employment retention of
91.8% for 2024 Q3->Q4 (implying an 8.2% separation rate that quarter) and 94.0% for 2019 Q3->Q4
(6.0%) -- statssa.gov.za/?p=19090. This notebook's rate for 2025-26 sits close to and slightly
above StatsSA's own 2024 figure, in the same direction as the deteriorating labour market
`discouraged_share` (notebook 01) already showed over the same window.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings(
    "ignore", category=UnicodeWarning
)  # QLFS string fields fall back to latin-1; expected, not a bug

# Set THESIS_DATA_ROOT to wherever you extracted the DataFirst downloads -- see data/README.md.
# No committed absolute path here: this notebook must run unmodified on a machine that isn't
# this thesis's own.
DATA_ROOT = Path(os.environ["THESIS_DATA_ROOT"])

QUARTERS = {
    "2025-Q2": DATA_ROOT / "qlfs-2025-q2-v1" / "qlfs-2025-q2-v1.dta",
    "2025-Q3": DATA_ROOT / "qlfs-2025-03" / "QLFS202503.dta",
    "2025-Q4": DATA_ROOT / "qlfs-2025-04" / "qlfs-2025-q4-v1.dta",
    "2026-Q1": DATA_ROOT / "qlfs-2026-q1-v1" / "qlfs-2026-q1-v1.dta",
}
LONG_TERM_CATEGORIES = [
    "Long-term unemployment (1 year and longer)",
    "Short-term unemployment (less than 1 year)",
]


def weighted_share(df: pd.DataFrame, flag_col: str, flag_value, weight_col: str = "Weight"):
    w = df[weight_col].to_numpy()
    is_flag = (df[flag_col] == flag_value).to_numpy().astype(float)
    p_hat = np.average(is_flag, weights=w)
    n = len(df)
    deff = (w**2).sum() * n / (w.sum() ** 2)
    se = np.sqrt(p_hat * (1 - p_hat) / (n / deff))
    return p_hat, se, n


results = {}
for quarter, path in QUARTERS.items():
    df = pd.read_stata(path, columns=["Long_term_unempl", "Weight"], convert_categoricals=True)
    sub = df[df["Long_term_unempl"].isin(LONG_TERM_CATEGORIES)]
    p_hat, se, n = weighted_share(sub, "Long_term_unempl", LONG_TERM_CATEGORIES[0])
    results[quarter] = {"n": n, "long_term_share": p_hat, "se": se}
    print(f"{quarter}: n={n:,}  long_term_share={p_hat:.4f}  se={se:.4f}")

results_df = pd.DataFrame(results).T

In [ ]:
headline_quarter = "2026-Q1"
headline = results_df.loc[headline_quarter]

moments_path = Path("../../data/moments.csv")
moments = pd.read_csv(moments_path)
moments["period"] = moments["period"].astype("object")
moments["source"] = moments["source"].astype("object")
row = moments["key"] == "long_term_share"
moments.loc[row, "value"] = round(float(headline["long_term_share"]), 4)
moments.loc[row, "standard_error"] = round(float(headline["se"]), 4)
moments.loc[row, "period"] = headline_quarter
moments.loc[row, "source"] = (
    "Statistics South Africa. Quarterly Labour Force Survey 2026: Q1 [dataset]. "
    "Cape Town: DataFirst [distributor]. QLFS Long_term_unempl variable, among the "
    "currently unemployed, person-weighted."
)
moments.loc[row, "provisional"] = False
moments.to_csv(moments_path, index=False)
moments[row]

## Part B: building the QLFS panel

Loads the same four quarters, this time keeping `UQNO` and `PERSONNO` to build a within-person
identifier and link consecutive quarters, plus `Q13GENDER`, `Q14AGE` and `Q15POPULATION` for the
demographic consistency check and `Stratum` for the design-variable audit and the household-
cluster bootstrap below.

In [ ]:
PANEL_COLS = ["UQNO", "PERSONNO", "Status", "Weight", "Q13GENDER", "Q14AGE", "Q15POPULATION"]
# The stratum field is spelled STRATUM in the 2025-Q2 extract and Stratum in the other three --
# confirmed by inspecting each file's own column list directly, not assumed from one file.
STRATUM_COL_BY_QUARTER = {
    "2025-Q2": "STRATUM",
    "2025-Q3": "Stratum",
    "2025-Q4": "Stratum",
    "2026-Q1": "Stratum",
}

panel_frames = {}
for quarter, path in QUARTERS.items():
    df = pd.read_stata(
        path,
        columns=[*PANEL_COLS, STRATUM_COL_BY_QUARTER[quarter]],
        convert_categoricals=True,
    )
    df = df.rename(columns={STRATUM_COL_BY_QUARTER[quarter]: "Stratum"})
    df["pid"] = df["UQNO"].astype(str) + "_" + df["PERSONNO"].astype(str)
    n_dup = df["pid"].duplicated().sum()
    if n_dup:
        raise ValueError(f"{quarter}: {n_dup} duplicate UQNO+PERSONNO identifiers -- not unique")
    panel_frames[quarter] = df.set_index("pid")
    print(f"{quarter}: {len(df):,} respondents, 0 duplicate person identifiers")

quarter_labels = list(QUARTERS.keys())
for t0, t1 in zip(quarter_labels[:-1], quarter_labels[1:], strict=True):
    overlap = panel_frames[t0].index.intersection(panel_frames[t1].index)
    print(f"{t0} -> {t1}: {len(overlap):,} matched of {len(panel_frames[t0]):,} in {t0}")

### Design-variable audit

Before trusting any resampling scheme, check what the raw files actually carry. Searched every
one of the four quarters' full column lists and Stata variable *labels* (not just names, in case
a design variable exists under a name that doesn't say so) for a rotation-group indicator, a
primary sampling unit (PSU), an enumeration-area/cluster code, or a wave indicator.

In [ ]:
DESIGN_KEYWORDS = ["rotat", "psu", "sample unit", "cluster", "wave", "panel"]

for quarter, path in QUARTERS.items():
    with pd.io.stata.StataReader(path) as reader:
        reader.read(1)  # populate the reader's own metadata before variable_labels() works
        labels = reader.variable_labels()
    name_hits = [c for c in labels if any(k in c.lower() for k in DESIGN_KEYWORDS)]
    label_hits = {c: v for c, v in labels.items() if any(k in v.lower() for k in DESIGN_KEYWORDS)}
    print(
        f"{quarter}: {len(labels)} variables; STRATUM present; rotation/PSU/cluster/wave "
        f"by name={name_hits or 'none'}, by label={label_hits or 'none'}"
    )

### The scale of the raw-linkage problem, over every matched person (not just the employed)

Before restricting to the employed-in-t0 population the separation rate actually needs,
reproduce the audit's own headline finding directly: how many of the *entire* matched
population in each quarter-pair have an unstable recorded gender or an implausible age change,
regardless of employment status. This is the audit trail for the "implausible matches" finding,
independent of what happens downstream.

In [ ]:
for t0, t1 in zip(quarter_labels[:-1], quarter_labels[1:], strict=True):
    a, b = panel_frames[t0], panel_frames[t1]
    common = a.index.intersection(b.index)
    a_c, b_c = a.loc[common], b.loc[common]

    gender_a = a_c["Q13GENDER"].astype(str).to_numpy()
    gender_b = b_c["Q13GENDER"].astype(str).to_numpy()
    gender_stable = gender_a == gender_b
    age_diff = pd.to_numeric(b_c["Q14AGE"], errors="coerce") - pd.to_numeric(
        a_c["Q14AGE"], errors="coerce"
    )
    age_ok = age_diff.isin([0, 1]).to_numpy()

    n_gender = int((~gender_stable).sum())
    n_age = int((~age_ok).sum())
    print(
        f"{t0} -> {t1}: {len(common):,} matched -- gender mismatches={n_gender}, "
        f"implausible age changes={n_age}"
    )

In [ ]:
# For each matched pair, keep everyone employed in t0, then split into two linkage tiers:
# "raw" (every UQNO+PERSONNO match, no further check) and "consistent" (raw, further requiring
# stable recorded gender and an age change of zero or one -- the two demographic fields every
# quarter carries). A household code reassigned to a different person between quarters produces
# a spurious transition under the raw linkage alone; these two checks catch the case where the
# reassignment is to someone of a different recorded gender or an implausible age jump, without
# needing a rotation-group or PSU field neither file actually has (see the design-variable audit
# above). Rejection counts below are restricted to people employed in t0, since that's the
# population the separation rate is actually computed over -- the cell above already reported
# the same two checks' rejection counts over the full matched population.
raw_sep_flags, raw_weights = [], []
consistent_sep_flags, consistent_weights = [], []
consistent_rows = []  # per-row records for the household-cluster bootstrap below

for t0, t1 in zip(quarter_labels[:-1], quarter_labels[1:], strict=True):
    a, b = panel_frames[t0], panel_frames[t1]
    common = a.index.intersection(b.index)
    a_c, b_c = a.loc[common], b.loc[common]

    employed_t0 = (a_c["Status"] == "Employed").to_numpy()
    separated = employed_t0 & (b_c["Status"] != "Employed").to_numpy()
    w = a_c["Weight"].to_numpy()

    gender_stable = (a_c["Q13GENDER"].astype(str) == b_c["Q13GENDER"].astype(str)).to_numpy()
    age_diff = pd.to_numeric(b_c["Q14AGE"], errors="coerce") - pd.to_numeric(
        a_c["Q14AGE"], errors="coerce"
    )
    age_ok = age_diff.isin([0, 1]).to_numpy()
    pop_stable = (a_c["Q15POPULATION"].astype(str) == b_c["Q15POPULATION"].astype(str)).to_numpy()
    consistent = gender_stable & age_ok

    n_gender_reject = int((employed_t0 & ~gender_stable).sum())
    n_age_reject = int((employed_t0 & ~age_ok).sum())
    n_pop_reject = int((employed_t0 & ~pop_stable).sum())
    n_raw = int(employed_t0.sum())
    n_consistent = int((employed_t0 & consistent).sum())
    attrition = 1 - n_consistent / n_raw

    raw_rate = np.average(separated[employed_t0].astype(float), weights=w[employed_t0])
    keep = employed_t0 & consistent
    consistent_rate = np.average(separated[keep].astype(float), weights=w[keep])

    print(
        f"{t0} -> {t1}, employed in {t0} only: n_raw={n_raw:,}  gender_mismatches="
        f"{n_gender_reject}  implausible_age_changes={n_age_reject}  (population_group_"
        f"mismatches={n_pop_reject}, not applied -- see markdown below)  n_consistent="
        f"{n_consistent:,}  attrition={attrition:.2%}"
    )
    print(f"    raw_rate={raw_rate:.4f}  consistent_rate={consistent_rate:.4f}")

    raw_sep_flags.append(separated[employed_t0])
    raw_weights.append(w[employed_t0])
    consistent_sep_flags.append(separated[keep])
    consistent_weights.append(w[keep])
    consistent_rows.append(
        pd.DataFrame(
            {
                "pair": f"{t0}->{t1}",
                "uqno": a_c["UQNO"].astype(str).to_numpy()[keep],
                "stratum": a_c["Stratum"].astype(str).to_numpy()[keep],
                "weight": w[keep],
                "separated": separated[keep].astype(float),
            }
        )
    )

raw_flags = np.concatenate(raw_sep_flags).astype(float)
raw_w = np.concatenate(raw_weights)
pooled_raw_quarterly = np.average(raw_flags, weights=raw_w)

consistent_flags = np.concatenate(consistent_sep_flags).astype(float)
consistent_w = np.concatenate(consistent_weights)
pooled_consistent_quarterly = np.average(consistent_flags, weights=consistent_w)

matched = pd.concat(consistent_rows, ignore_index=True)
matched["cluster"] = matched["pair"] + "::" + matched["uqno"]

print(f"\npooled RAW quarterly separation rate: {pooled_raw_quarterly:.10f}")
print(f"pooled CONSISTENT quarterly separation rate: {pooled_consistent_quarterly:.10f}")
print(
    f"consistent sample: {len(matched):,} person-quarters in {matched['cluster'].nunique():,} "
    f"distinct households, across {matched['stratum'].nunique()} strata"
)

### Why population group is checked but not applied as a third filter

`Q15POPULATION` is available and consistently coded (the same four categories) in every quarter,
so it was checked for stability the same way gender and age were: 37, 17 and 33 mismatches
across the three pairs respectively, out of roughly 10,500-10,700 employed-t0 records per pair --
under 0.3 per cent. Adding it as a third filter moves the pooled quarterly rate from
0.0861488108 to 0.0861345569, a change in the fifth decimal place, and the monthly rate from
0.0295827815 to 0.0295777362 -- both round to the same 0.0296 either way. Reported here for
completeness rather than silently checked and dropped, but not applied as a filter: the frozen
central estimate below uses gender and age only, matching the change this correction is actually
about (removing the raw linkage's spurious transitions), not chasing a fourth-significant-figure
difference that changes nothing downstream.

In [ ]:
# Household-cluster bootstrap, stratified by Stratum: resample whole households (cluster =
# pair::UQNO, so a household re-appearing in a later pair is a distinct cluster each time it's
# matched, never double-counted as one unit) with replacement, independently within each
# stratum so every replicate keeps each stratum's own household count -- rather than resampling
# individual person-quarters as though they were independent draws, which understates variance
# whenever a household contributes more than one matched person.
cluster_stratum = matched.groupby("cluster")["stratum"].first()
clusters_by_stratum = {
    s: cluster_stratum.index[cluster_stratum == s].to_numpy() for s in cluster_stratum.unique()
}
rowidx_by_cluster = matched.groupby("cluster").indices  # cluster -> array of row positions

sep_arr = matched["separated"].to_numpy()
w_arr = matched["weight"].to_numpy()

rng = np.random.default_rng(42)
n_boot = 2000
boot_quarterly = np.empty(n_boot)
for i in range(n_boot):
    row_idx = []
    for stratum_clusters in clusters_by_stratum.values():
        draw = rng.choice(stratum_clusters, size=len(stratum_clusters), replace=True)
        for cluster in draw:
            row_idx.append(rowidx_by_cluster[cluster])
    row_idx = np.concatenate(row_idx)
    boot_quarterly[i] = np.average(sep_arr[row_idx], weights=w_arr[row_idx])

# Same quarterly-to-monthly compounding convention as rho_A (DECISIONS.md's AR(1) note):
# (1 - monthly)**3 = (1 - quarterly).
central_monthly_rate = 1 - (1 - pooled_consistent_quarterly) ** (1 / 3)
raw_monthly_rate = 1 - (1 - pooled_raw_quarterly) ** (1 / 3)
boot_monthly = 1 - (1 - boot_quarterly) ** (1 / 3)
quarterly_se = boot_quarterly.std(ddof=1)
monthly_se = boot_monthly.std(ddof=1)
ci_lo, ci_hi = np.percentile(boot_monthly, [2.5, 97.5])
fallback_ratio = central_monthly_rate / 0.0048

print(
    f"central (consistent) quarterly rate: {pooled_consistent_quarterly:.10f} "
    f"(household-cluster bootstrap SE {quarterly_se:.6f})"
)
print(
    f"central (consistent) monthly rate:   {central_monthly_rate:.10f} "
    f"(household-cluster bootstrap SE {monthly_se:.6f})"
)
print(f"95% household-cluster bootstrap CI (monthly): [{ci_lo:.6f}, {ci_hi:.6f}]")
print(
    f"\nraw-link monthly rate (robustness upper bound, no consistency filter): "
    f"{raw_monthly_rate:.10f}"
)
print(
    f"\nMiyamoto (2011)'s Japanese fallback: 0.0048 -- the central estimate is "
    f"{fallback_ratio:.1f}x higher"
)

## Result

**long_term_share**: written to `moments.csv` above, most recent quarter as headline (same
convention as notebook 01).

**Separation rate**: the central estimate, after requiring stable recorded gender and a
plausible age change (zero or one year) across each matched pair, is **2.96% monthly**
(unrounded 0.0295827815, household-cluster bootstrap 95% CI printed above) -- this is what
Task 4 of the verification remediation plan calls "an approximate QLFS panel estimate," not "the
official SA separation rate": it doesn't reproduce StatsSA's own panel non-response adjustment,
because the four quarterly extracts on disk carry no rotation-group or PSU field to adjust with
(confirmed above, not assumed). The uncorrected raw-link rate, 3.17% monthly (unrounded
0.0316956564), is kept as an upper robustness bound, not the central value -- it's inflated by
the spurious transitions the gender/age consistency check exists to catch.

Both figures are propagated into `configs/baseline.yaml` and every config that's supposed to
share its economics (`frictionless.yaml`, `trace_demo.yaml`, both `scarce_vacancies*.yaml`) as
part of this same remediation task, replacing the still-present `separation_rate: 0.0317`
placeholder in those files with the corrected `0.0296` central estimate --
`tests/test_configs.py` asserts they all agree. See DECISIONS.md, "The QLFS separation-rate
linkage needed a demographic consistency check, and the design variables needed auditing before
trusting a plain bootstrap," for the full write-up, including why population group was checked
but not applied as a filter, and both re-checks (`trace_demo.yaml`'s trajectory, the scarce-
vacancy smoke test) required whenever this rate changes.